# Traditional RAG — truy xuất dữ liệu bất động sản

**A. Lập chỉ mục:** Load dữ liệu → kiểm tra và làm sạch → chia chunk → khởi tạo embedding → chuyển chunk thành vector → lưu nội dung.

**B. Truy vấn:** Test câu hỏi → tìm và hiển thị các tin liên quan.

Mã xử lý nằm trực tiếp trong từng bước, kèm đầu ra để quan sát.


## Chuẩn bị môi trường

Từ thư mục gốc:

```bash
.venv/bin/python -m pip install -r model/TraditionalRAG/requirements.txt
docker compose up -d vector_db
.venv/bin/python -m jupyter lab
```

Chọn kernel có dependency đã cài. Embedding E5 được tải ở lần đầu chạy. PostgreSQL đọc thông tin kết nối từ `.env`.

Chạy các cell từ trên xuống bằng Shift+Enter: cell import bên dưới → bước 1. Load dữ liệu → bước 2. Kiểm tra và làm sạch dữ liệu. Sau khi restart kernel, cần chạy lại các cell trước đó; đầu ra lưu sẵn trong notebook không khởi tạo biến trong kernel.


In [1]:
# Tìm thư mục gốc để đọc .env và lưu kết quả đúng chỗ dù mở notebook từ thư mục mô hình.

from pathlib import Path
from itertools import islice
from time import perf_counter
import csv
import hashlib
import json
import math
import re
from IPython.display import display

# Tìm thư mục gốc dự án: đi ngược từ thư mục hiện tại lên các thư mục cha,
# dừng ở thư mục có model/ và app/, không cần CSV cục bộ.
working_directory = Path.cwd().resolve()
root = next(
    (
        directory
        for directory in (working_directory, *working_directory.parents)
        if (directory / "model").is_dir() and (directory / "app").is_dir()
    ),
    None,
)
if root is None:
    raise RuntimeError("Mở notebook trong thư mục dự án hoặc thư mục mô hình")


## 1. Load dữ liệu

Mặc định đọc 10 bản ghi đầu từ [tinixai/vietnam-real-estates](https://huggingface.co/datasets/tinixai/vietnam-real-estates) bằng `streaming=True` và hiển thị dạng bảng trước khi làm sạch. Link đã nằm trong cell bên dưới; không cần cấu hình `.env` hoặc tải CSV cục bộ. Có thể ghi đè bằng `DATA_URL` (đặt rỗng để dùng CSV cục bộ).


In [ ]:
# Đọc dữ liệu nguồn theo luồng; chỉ các bản ghi mẫu mới được giữ trong bộ nhớ.

import os
from contextlib import ExitStack, closing
from io import TextIOWrapper
from urllib.parse import urlsplit
from urllib.request import Request, urlopen
from dotenv import load_dotenv

# Cell import phải chạy trước để có biến root.
if "root" not in globals():
    raise RuntimeError("Chạy cell import trong phần Chuẩn bị môi trường trước bước 1. Load dữ liệu.")

from html import escape
from IPython.display import HTML

# Đọc 10 bản ghi đầu theo luồng; utf-8-sig bỏ BOM nếu dùng CSV.
# Đọc cấu hình nếu có; override=False giữ ưu tiên cho biến môi trường đã đặt bên ngoài.
load_dotenv(root / ".env", override=False)
# Mặc định đọc streaming từ Hugging Face; DATA_URL có thể ghi đè nguồn.
data_url = os.getenv(
    "DATA_URL",
    "https://huggingface.co/datasets/tinixai/vietnam-real-estates",
).strip()
if data_url and (urlsplit(data_url).scheme not in {"http", "https"} or not urlsplit(data_url).hostname):
    raise ValueError("DATA_URL phải là link dataset Hugging Face hoặc link HTTP(S) tải CSV")
# Tách tên máy chủ và đường dẫn để phân biệt trang dataset Hugging Face với link CSV.
parsed_data_url = urlsplit(data_url)
data_segments = parsed_data_url.path.strip("/").split("/")
# ExitStack quản lý các tài nguyên mở trong cell: đủ số dòng hoặc có lỗi đều đóng file/kết nối.
# closing() đóng bộ lặp streaming khi dừng sớm, tránh giữ tài nguyên cho phần chưa đọc.
with ExitStack() as stack:
    if parsed_data_url.hostname == "huggingface.co" and len(data_segments) == 3 and data_segments[0] == "datasets":
        from datasets import load_dataset
        # Lấy mã repo namespace/tên-dataset từ link; train là tập dữ liệu đang dùng.
        # streaming=True đọc Parquet theo lô khi lặp, không tải toàn bộ dataset xuống máy.
        dataset = load_dataset("/".join(data_segments[1:]), split="train", streaming=True)
        iterator = stack.enter_context(closing(iter(dataset)))
        # Parquet có số và null; đổi thành chuỗi/rỗng để dùng chung bước làm sạch với CSV.
        reader = ({key: "" if value is None else str(value) for key, value in record.items()} for record in iterator)
        columns = list(dataset.column_names or [])
    else:
        if data_url:
            # Với CSV qua HTTP, đọc nội dung tải trực tiếp; identity yêu cầu dữ liệu không nén HTTP.
            response = stack.enter_context(urlopen(Request(data_url, headers={"Accept-Encoding": "identity"}), timeout=60))
            # Link chia sẻ có thể trả trang HTML; báo lỗi sớm thay vì hiểu nhầm trang web là dữ liệu.
            if response.headers.get_content_type() == "text/html":
                raise ValueError("DATA_URL trả về trang HTML; cần link dataset Hugging Face hoặc CSV trực tiếp")
            handle = stack.enter_context(TextIOWrapper(response, encoding="utf-8-sig", newline=""))
        else:
            handle = stack.enter_context(open(root / "data" / "vietnam-real-estates.csv", encoding="utf-8-sig", newline=""))
        # Mỗi bản ghi CSV thành dict theo tên cột; utf-8-sig ở trên bỏ BOM nếu file có BOM.
        reader = csv.DictReader(handle)
        columns = reader.fieldnames or []
    # Chỉ giữ 10 bản ghi để xem mẫu; không chuyển toàn bộ bộ lặp streaming thành list.
    raw_rows = list(islice(reader, 10))
print("Số dòng đã load:", len(raw_rows))
print("Các cột:", columns)
# Dựng bảng HTML thủ công (escape để tránh lỗi hiển thị khi dữ liệu có ký tự < > &),
# cho phép ô xuống dòng để đọc được mô tả dài.
header_html = "".join(f"<th>{escape(column)}</th>" for column in columns)
rows_html = "".join(
    "<tr>" + "".join(
        f'<td style="min-width: 100px; max-width: 300px; white-space: normal; overflow-wrap: anywhere; vertical-align: top;">{escape(row.get(column) or "")}</td>'
        for column in columns
    ) + "</tr>"
    for row in raw_rows
)
display(HTML(
    '<div style="overflow-x: auto;"><table class="dataframe">'
    f"<thead><tr>{header_html}</tr></thead><tbody>{rows_html}</tbody></table></div>"
))


## 2. Kiểm tra và làm sạch dữ liệu

Kiểm tra các cột, chuẩn hóa khoảng trắng và trường số. Giữ ID dòng CSV và dấu tiếng Việt.

Nếu chạy riêng cell này khi chưa load dữ liệu, cell tự đọc 10 dòng đầu CSV và import các thư viện cần thiết.


In [ ]:
# Đọc dữ liệu nguồn theo luồng; chỉ các bản ghi mẫu mới được giữ trong bộ nhớ.

import os
from contextlib import ExitStack, closing
from io import TextIOWrapper
from urllib.parse import urlsplit
from urllib.request import Request, urlopen
from dotenv import load_dotenv

from pathlib import Path
from itertools import islice
import csv
import math
import re
from IPython.display import display

# Cho phép chạy riêng cell này sau khi mở notebook hoặc restart kernel.
if not {"columns", "raw_rows"} <= globals().keys():
    working_directory = Path.cwd().resolve()
    root = next(
        (
            directory
            for directory in (working_directory, *working_directory.parents)
            if (directory / "model").is_dir() and (directory / "app").is_dir()
        ),
        None,
    )
    if root is None:
        raise RuntimeError("Mở notebook trong thư mục dự án hoặc thư mục mô hình")
    # Đọc cấu hình nếu có; override=False giữ ưu tiên cho biến môi trường đã đặt bên ngoài.
    load_dotenv(root / ".env", override=False)
    # Mặc định đọc streaming từ Hugging Face; DATA_URL có thể ghi đè nguồn.
    data_url = os.getenv(
        "DATA_URL",
        "https://huggingface.co/datasets/tinixai/vietnam-real-estates",
    ).strip()
    if data_url and (urlsplit(data_url).scheme not in {"http", "https"} or not urlsplit(data_url).hostname):
        raise ValueError("DATA_URL phải là link dataset Hugging Face hoặc link HTTP(S) tải CSV")
    # Tách tên máy chủ và đường dẫn để phân biệt trang dataset Hugging Face với link CSV.
    parsed_data_url = urlsplit(data_url)
    data_segments = parsed_data_url.path.strip("/").split("/")
    # ExitStack quản lý các tài nguyên mở trong cell: đủ số dòng hoặc có lỗi đều đóng file/kết nối.
    # closing() đóng bộ lặp streaming khi dừng sớm, tránh giữ tài nguyên cho phần chưa đọc.
    with ExitStack() as stack:
        if parsed_data_url.hostname == "huggingface.co" and len(data_segments) == 3 and data_segments[0] == "datasets":
            from datasets import load_dataset
            # Lấy mã repo namespace/tên-dataset từ link; train là tập dữ liệu đang dùng.
            # streaming=True đọc Parquet theo lô khi lặp, không tải toàn bộ dataset xuống máy.
            dataset = load_dataset("/".join(data_segments[1:]), split="train", streaming=True)
            iterator = stack.enter_context(closing(iter(dataset)))
            # Parquet có số và null; đổi thành chuỗi/rỗng để dùng chung bước làm sạch với CSV.
            reader = ({key: "" if value is None else str(value) for key, value in record.items()} for record in iterator)
            columns = list(dataset.column_names or [])
        else:
            if data_url:
                # Với CSV qua HTTP, đọc nội dung tải trực tiếp; identity yêu cầu dữ liệu không nén HTTP.
                response = stack.enter_context(urlopen(Request(data_url, headers={"Accept-Encoding": "identity"}), timeout=60))
                # Link chia sẻ có thể trả trang HTML; báo lỗi sớm thay vì hiểu nhầm trang web là dữ liệu.
                if response.headers.get_content_type() == "text/html":
                    raise ValueError("DATA_URL trả về trang HTML; cần link dataset Hugging Face hoặc CSV trực tiếp")
                handle = stack.enter_context(TextIOWrapper(response, encoding="utf-8-sig", newline=""))
            else:
                handle = stack.enter_context(open(root / "data" / "vietnam-real-estates.csv", encoding="utf-8-sig", newline=""))
            # Mỗi bản ghi CSV thành dict theo tên cột; utf-8-sig ở trên bỏ BOM nếu file có BOM.
            reader = csv.DictReader(handle)
            columns = reader.fieldnames or []
        # Chỉ giữ 10 bản ghi để xem mẫu; không chuyển toàn bộ bộ lặp streaming thành list.
        raw_rows = list(islice(reader, 10))
    print("Đã tự load", len(raw_rows), "dòng CSV để làm sạch.")

# Các cột bắt buộc phải có trong CSV.
# Kiểm tra các cột mà bước truy xuất cần trước khi xử lý dữ liệu.
required = {"name", "description", "province_name", "district_name", "price", "area"}
missing_columns = required - set(columns)
if missing_columns:
    raise ValueError(f"CSV thiếu cột: {sorted(missing_columns)}")
# Mỗi dòng CSV thành một tài liệu; ID là số thứ tự dòng (bắt đầu từ 1) để truy ngược về nguồn.
documents = []
for source_id, raw in enumerate(raw_rows, 1):
    # Gộp mọi khoảng trắng liên tiếp (kể cả xuống dòng) thành một dấu cách và cắt hai đầu.
    row = {key: re.sub("\\s+", " ", value or "").strip() for key, value in raw.items() if key}
    # Bỏ tin không có cả tiêu đề lẫn mô tả vì không có gì để tìm kiếm.
    if not row["name"] and (not row["description"]):
        continue
    metadata = dict(row)
    # Chuyển trường số sang float; giá trị rỗng, sai định dạng, NaN/inf hoặc âm thành None.
    for field in ("price", "area", "bedroom_count", "bathroom_count", "floor_count"):
        try:
            value = float(row.get(field, ""))
            metadata[field] = value if math.isfinite(value) and value >= 0 else None
        except ValueError:
            metadata[field] = None
    documents.append(
        dict(
            id=str(source_id), title=row["name"], description=row["description"], metadata=metadata
        )
    )
if not documents:
    raise ValueError("Không có tài liệu hợp lệ")
# Thống kê nhanh kết quả làm sạch và xem thử tài liệu đầu tiên.
print("Tin hợp lệ:", len(documents), "| tin bị bỏ:", len(raw_rows) - len(documents))
print(
    "Trường số thiếu/không hợp lệ:",
    {
        field: sum((doc["metadata"][field] is None for doc in documents))
        for field in ("price", "area", "bedroom_count", "bathroom_count", "floor_count")
    },
)
display(documents[0])


## 3. Chia chunk

Dùng tokenizer của E5 để chia từng tin thành các đoạn tối đa 384 token, chồng lấp 48 token. Mỗi đoạn giữ nội dung, thuộc tính và ID nguồn.


In [4]:
# Chia mô tả tin đăng thành các đoạn (chunk) tối đa 384 token, chồng lấp 48 token.

from transformers import AutoTokenizer

# Dùng đúng tokenizer của model embedding để đếm token chính xác.
# Cần fast tokenizer vì chỉ bản này trả về offset_mapping (vị trí ký tự của từng token).
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")
if not tokenizer.is_fast:
    raise ValueError("Cần fast tokenizer")
chunks = []
for doc in documents:
    meta = doc["metadata"]
    # Header gồm tiêu đề và các thuộc tính chính; được lặp lại ở mọi chunk của cùng một tin
    # để mỗi chunk vẫn đủ ngữ cảnh khi được truy xuất riêng lẻ.
    header = "\n".join(
        [f"Tiêu đề: {doc['title']}"]
        + [
            f"{field}: {meta[field]}"
            for field in ("province_name", "district_name", "ward_name", "street_name")
            + ("property_type_name", "project_name", "price", "area")
            if meta.get(field) is not None and meta.get(field) != ""
        ]
    )
    # Số token còn lại cho phần mô tả trong mỗi chunk: 384 trừ header, tiền tố "passage: "
    # và 4 token dự phòng cho token đặc biệt.
    budget = 384 - len(tokenizer.encode("passage: " + header + "\n")) - 4
    if not 0 <= 48 < budget:
        raise ValueError("Header quá dài hoặc overlap quá lớn")
    body = doc["description"]
    # Vị trí (ký tự bắt đầu, ký tự kết thúc) của từng token trong mô tả,
    # dùng để cắt văn bản gốc đúng ranh giới token mà không làm hỏng dấu tiếng Việt.
    offsets = tokenizer(body, add_special_tokens=False, return_offsets_mapping=True)[
        "offset_mapping"
    ]
    # Cửa sổ trượt: start/end là chỉ số token của chunk hiện tại, index là số thứ tự chunk.
    start, index = (0, 0)
    while True:
        # Lấy tối đa budget token; mô tả rỗng thì chunk chỉ có header.
        end = min(start + budget, len(offsets))
        text = header + "\n" + (body[offsets[start][0] : offsets[end - 1][1]] if offsets else "")
        # Tokenize lại cả chunk có thể ra nhiều token hơn ước tính,
        # nên bớt dần từng token cho đến khi vừa giới hạn 384.
        while len(tokenizer.encode("passage: " + text)) > 384:
            end -= 1
            if end <= start:
                raise ValueError("Không đủ token budget")
            text = header + "\n" + body[offsets[start][0] : offsets[end - 1][1]]
        # Lưu chunk kèm metadata của tin và vị trí token để biết chunk lấy từ đoạn nào.
        chunks.append(
            dict(
                id=f"{doc['id']}:{index}",
                listing_id=doc["id"],
                text=text,
                metadata={
                    **meta,
                    "source_ids": [doc["id"]],
                    "chunk_index": index,
                    "token_start": start,
                    "token_end": end,
                },
            )
        )
        # Đã phủ hết mô tả thì sang tin tiếp theo.
        if end >= len(offsets):
            break
        if end - start <= 48:
            raise ValueError("Overlap quá lớn sau khi tokenize lại")
        # Chunk sau bắt đầu lùi lại 48 token để chồng lấp, tránh cắt đứt ý ở ranh giới chunk.
        start, index = (end - 48, index + 1)
# Kiểm tra lại không chunk nào vượt 384 token.
token_counts = [len(tokenizer.encode("passage: " + item["text"])) for item in chunks]
assert max(token_counts) <= 384
print("Số tin:", len(documents), "| số chunk:", len(chunks), "| token lớn nhất:", max(token_counts))
display(chunks[0])


/home/ngtukien/Projects/Vietnam-Real-Estates/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Số tin: 10 | số chunk: 10 | token lớn nhất: 256


{'id': '1:0',
 'listing_id': '1',
 'text': 'Tiêu đề: Bán nhanh nhà 6 tầng thang máy, ngoạ long cầu diễn, ôtô đỗ cổng, giá: 7.45 tỷ, lh [phone_number]\nprovince_name: Hà Nội\ndistrict_name: Bắc Từ Liêm\nward_name: Phúc Diễn\nstreet_name: Ngọa Long\nproperty_type_name: Nhà\nprice: 7450000000.0\narea: 37.0\n- Vị trí siêu đẹp ô tô đỗ cổng, khu vực đang phát triển mạnh, tiện ích xung quanh đầy đủ, Gần chợ, Gần trường học các cấp, Bãi gửi xe, Gần trường đại học công nghiệp. - Nhà xây mới thiết kế hiện đại tặng full nội thất, 4 phòng ngủ rộng, công năng sử dụng đầy đủ. - Giao thông thuận tiện đi lại kết nối với các tuyến đường lớn, Đường Văn Tiến Dũng, Cầu Diễn, Ql32, Xuân Phương, Trịnh Văn Bô. - Pháp lý sổ đỏ chính chủ. - Giá: 7.45 tỷ (có thương lượng) + LH Em Vân [phone_number] để đi xem nhà thực tế.',
 'metadata': {'name': 'Bán nhanh nhà 6 tầng thang máy, ngoạ long cầu diễn, ôtô đỗ cổng, giá: 7.45 tỷ, lh [phone_number]',
  'description': '- Vị trí siêu đẹp ô tô đỗ cổng, khu vực đang phát t

## 4. Khởi tạo mô hình embedding

E5 chuyển văn bản thành vector để so sánh độ liên quan. Bước này tải model embedding đã huấn luyện, không huấn luyện model mới từ CSV.


In [5]:
# Khởi tạo E5 đã huấn luyện sẵn để biến văn bản tiếng Việt thành vector; notebook không huấn luyện lại model.

from sentence_transformers import SentenceTransformer
import numpy as np

# Tải model E5 đã huấn luyện sẵn (lần đầu sẽ tải từ Hugging Face) và chạy trên CPU.
encoder = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")
# Model phải nhận được chunk dài 384 token mà không bị cắt.
assert 384 <= encoder.max_seq_length
# Số chiều vector; bản sentence-transformers mới đổi tên hàm nên thử tên mới trước.
dimension = (
    getattr(encoder, "get_embedding_dimension", None) or encoder.get_sentence_embedding_dimension
)()
print(
    "Embedding:",
    "intfloat/multilingual-e5-small",
    "| chiều:",
    dimension,
    "| giới hạn:",
    encoder.max_seq_length,
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14525.54it/s]

Embedding: intfloat/multilingual-e5-small | chiều: 384 | giới hạn: 512


## 5. Chuyển chunk thành vector

Thêm tiền tố `passage:` và chuẩn hóa embedding. Hiển thị kích thước ma trận vector.


In [6]:
# Mỗi chunk được biến thành một vector để sau này tìm văn bản có nghĩa gần với câu hỏi.

started = perf_counter()
# E5 yêu cầu tiền tố "passage: " cho văn bản được lưu (câu hỏi dùng "query: ").
# normalize_embeddings=True đưa vector về độ dài 1 để so sánh bằng cosine.
embeddings = encoder.encode(
    ["passage: " + item["text"] for item in chunks],
    normalize_embeddings=True,
).astype(np.float32)
# Kiểm tra: đủ vector, không có NaN/inf và mọi vector đã được chuẩn hóa.
assert embeddings.shape == (len(chunks), dimension)
assert np.isfinite(embeddings).all()
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=0.0001)
print("Shape:", embeddings.shape, "| thời gian:", round(perf_counter() - started, 2), "giây")
print("8 giá trị đầu:", embeddings[0][:8])


Shape: (10, 384) | thời gian: 0.29 giây
8 giá trị đầu: [ 0.01654588 -0.02419186 -0.0187199   0.00344613  0.09123275 -0.00713979
 -0.00148701  0.05045026]


## 6. Lưu nội dung

Lưu chunk, vector và metadata vào PostgreSQL/pgvector. Collection được xác định từ các chunk và tokenizer; chạy lại dùng upsert để tránh trùng dữ liệu.


In [7]:
# Mở PostgreSQL để lưu/tìm vector; các bảng của mô hình sẽ dùng cùng kết nối conn.

# Tên collection là mã băm của nội dung chunk và tokenizer:
# dữ liệu không đổi thì chạy lại vẫn ghi vào cùng collection, dữ liệu đổi thì tạo collection mới.
collection = hashlib.sha256(
    json.dumps([chunks, tokenizer.name_or_path], ensure_ascii=False, sort_keys=True).encode()
).hexdigest()[:24]
print("Collection:", collection)
import os
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector

# Đọc thông tin kết nối PostgreSQL từ file .env ở thư mục gốc.
load_dotenv(root / ".env", override=False)
# Chạy lại cell thì đóng kết nối cũ trước khi mở kết nối mới.
if "conn" in globals() and not conn.closed:
    conn.close()
# autocommit lưu từng lệnh ngay; các khối conn.transaction() bên dưới vẫn gom thao tác liên quan thành một lần ghi.
conn = psycopg.connect(
    host=os.getenv("POSTGRES_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_PORT", "5432"),
    dbname=os.getenv("POSTGRES_DB", "real_estates"),
    user=os.getenv("POSTGRES_USER", "real_estates"),
    password=os.getenv("POSTGRES_PASSWORD", "local_dev_password"),
    autocommit=True,
    connect_timeout=5,
)
# Bật extension pgvector và đăng ký kiểu vector để truyền numpy array trực tiếp.
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
register_vector(conn)
from psycopg import sql

if not isinstance(dimension, int) or dimension <= 0:
    raise ValueError("Invalid dimension")
# Mỗi số chiều embedding dùng một bảng riêng vì cột vector cố định số chiều.
# Khóa chính (collection, kind, id) giúp upsert không tạo bản ghi trùng.
vector_table_name = f"notebook_vectors_{dimension}"
conn.execute(
    sql.SQL(
        "CREATE TABLE IF NOT EXISTS {}(collection text,kind text,id text,listing_id text,\n        content text NOT NULL,metadata jsonb NOT NULL,embedding vector({}) NOT NULL,PRIMARY KEY(collection,kind,id))"
    ).format(sql.Identifier(vector_table_name), sql.Literal(dimension))
)
from psycopg.types.json import Jsonb

if len(chunks) != len(embeddings):
    raise ValueError("Số chunk và vector không khớp")
# Upsert: chunk đã tồn tại thì cập nhật nội dung, metadata và vector thay vì báo lỗi trùng khóa.
statement = sql.SQL(
    "INSERT INTO {} VALUES(%s,%s,%s,%s,%s,%s,%s) ON CONFLICT(collection,kind,id)\n        DO UPDATE SET listing_id=excluded.listing_id,content=excluded.content,metadata=excluded.metadata,embedding=excluded.embedding"
).format(sql.Identifier(vector_table_name))
# Ghi toàn bộ chunk trong một transaction: lỗi giữa chừng thì không ghi gì.
with conn.transaction(), conn.cursor() as cursor:
    cursor.executemany(
        statement,
        [
            (
                collection,
                "listing_chunk",
                c["id"],
                c.get("listing_id"),
                c["text"],
                Jsonb(c["metadata"]),
                v,
            )
            for c, v in zip(chunks, embeddings)
        ],
    )
# Đếm lại số chunk trong collection để xác nhận đã lưu đủ.
count = conn.execute(
    sql.SQL("SELECT count(*) FROM {} WHERE collection=%s AND kind=%s").format(
        sql.Identifier(vector_table_name)
    ),
    (collection, "listing_chunk"),
).fetchone()[0]
print("Bảng:", vector_table_name, "| chunk đã lưu:", count)
assert count == len(chunks)


Collection: cd82d0df7dc9eb9055dc93eb
Bảng: notebook_vectors_384 | chunk đã lưu: 10


# B. Truy vấn

## 7. Test

Sửa câu hỏi ngay trong cell này rồi chạy lại. Dùng cùng model embedding với dữ liệu, tìm theo cosine và hiển thị tối đa 5 tin khác nhau; mỗi tin dùng chunk liên quan nhất.


In [8]:
# Traditional RAG xếp hạng bằng độ gần nghĩa; một tin chỉ giữ chunk tốt nhất trước khi lấy top 5.

from psycopg import sql
from psycopg.rows import dict_row

# Câu hỏi cần tìm; sửa ở đây rồi chạy lại cell.
query = "Nhà Hà Nội gần trường, có thang máy"
# Mã hóa câu hỏi bằng cùng model, dùng tiền tố "query: " theo quy ước của E5.
query_vector = encoder.encode(["query: " + query], normalize_embeddings=True).astype(np.float32)[0]
assert len(query_vector) == dimension

# <=> là khoảng cách cosine của pgvector; độ tương đồng = 1 - khoảng cách.
# DISTINCT ON (listing_id) chỉ giữ chunk gần nhất của mỗi tin,
# sau đó sắp xếp theo điểm và lấy 5 tin khác nhau.
statement = sql.SQL("""
    WITH nearest_chunks AS (
        SELECT DISTINCT ON (listing_id)
            id, listing_id, content, metadata, 1 - (embedding <=> %s) AS score
        FROM {}
        WHERE collection = %s AND kind = 'listing_chunk'
        ORDER BY listing_id, embedding <=> %s, id
    )
    SELECT id, listing_id, content, metadata, score
    FROM nearest_chunks
    ORDER BY score DESC, id
    LIMIT 5
""").format(sql.Identifier(vector_table_name))
# dict_row trả kết quả dạng dict để truy cập theo tên cột.
with conn.cursor(row_factory=dict_row) as cursor:
    cursor.execute(statement, (query_vector, collection, query_vector))
    hits = cursor.fetchall()

print("Câu hỏi:", query)
print("Số tin tìm được:", len(hits))
if not hits:
    print("Không tìm thấy dữ liệu phù hợp.")
for hit in hits:
    print("Tin:", hit["listing_id"], "| chunk:", hit["id"], "| cosine:", round(hit["score"], 4))
    print(hit["content"], "\n")


Câu hỏi: Nhà Hà Nội gần trường, có thang máy
Số tin tìm được: 5
Tin: 1 | chunk: 1:0 | cosine: 0.873
Tiêu đề: Bán nhanh nhà 6 tầng thang máy, ngoạ long cầu diễn, ôtô đỗ cổng, giá: 7.45 tỷ, lh [phone_number]
province_name: Hà Nội
district_name: Bắc Từ Liêm
ward_name: Phúc Diễn
street_name: Ngọa Long
property_type_name: Nhà
price: 7450000000.0
area: 37.0
- Vị trí siêu đẹp ô tô đỗ cổng, khu vực đang phát triển mạnh, tiện ích xung quanh đầy đủ, Gần chợ, Gần trường học các cấp, Bãi gửi xe, Gần trường đại học công nghiệp. - Nhà xây mới thiết kế hiện đại tặng full nội thất, 4 phòng ngủ rộng, công năng sử dụng đầy đủ. - Giao thông thuận tiện đi lại kết nối với các tuyến đường lớn, Đường Văn Tiến Dũng, Cầu Diễn, Ql32, Xuân Phương, Trịnh Văn Bô. - Pháp lý sổ đỏ chính chủ. - Giá: 7.45 tỷ (có thương lượng) + LH Em Vân [phone_number] để đi xem nhà thực tế. 

Tin: 2 | chunk: 2:0 | cosine: 0.8311
Tiêu đề: Bán gấp nhà mặt tiền 4 tầng khu phân lô Tạ Quang Bửu
province_name: Hồ Chí Minh
district_name: 8


### Giới hạn

Notebook minh họa phần truy xuất trong Traditional RAG. Kết quả là các tin và đoạn văn được tìm thấy. Dữ liệu đầu CSV chưa đại diện toàn thị trường; chưa có nhãn đánh giá chất lượng truy xuất. Kết nối được giữ để chạy lại bước Test; đóng kết nối hoặc dừng kernel khi kết thúc.
